# Tech / Architecture — Delta Lake and Apache Iceberg
**Day 11 — Modern Data Formats**

---

<div style="padding:15px;border-left:8px solid #f39c12;background:#fef9e7;border-radius:4px;">
<strong>Core Insight:</strong> Traditional data lakes are immutable append-only files (Parquet on S3/HDFS). Delta Lake and Iceberg add ACID transactions, schema evolution, time travel, and efficient upserts to data lakes. They solve the 'lakehouse' problem: you want S3/HDFS storage scaling and costs but database-level reliability and correctness.
</div>

## 1. The Lakehouse Problem: Why Parquet is Not Enough

**Traditional Parquet on S3:**
- No ACID — two writers can corrupt data
- No deletes or updates — append only (or full rewrite)
- No schema enforcement — silently accepts wrong schemas
- No time travel — can't query yesterday's data
- Slow upserts — must rewrite entire partition for CDC

**Delta Lake / Apache Iceberg:**
- ACID transactions (serializable isolation)
- Row-level updates and deletes (critical for GDPR erasure requests)
- Schema evolution + enforcement
- Time travel — query data as of any timestamp
- Efficient MERGE for CDC (change data capture)

## 2. Delta Lake Core Architecture

Delta Lake operates by maintaining a specific directory structure on top of your existing S3/HDFS storage.

```text
s3://my-bucket/server-metrics/
├── _delta_log/                <-- The Transaction Log (JSON + Checkpoint Parquet files)
│   ├── 00000000000000000000.json
│   ├── 00000000000000000001.json
│   └── 00000000000000000002.json
├── part-00000-xyz.parquet    <-- The actual data files
├── part-00001-xyz.parquet
└── part-00002-abc.parquet
```

**The `_delta_log` IS the database.** Readers first consult the log to discover which Parquet files exactly constitute the current, valid snapshot of the table. Writers write new Parquet files, then optimistically attempt to add a new `.json` commit to the log. If successful, the files become formally part of the table.

In [ ]:
class MockDeltaTable:
    """Simulates the PySpark DeltaTable API for educational purposes."""
    def __init__(self, path):
        self.path = path
        print(f"[Delta] Connecting to Delta Table at {path}")
        print(f"[Delta] Reading _delta_log to ascertain current snapshot...")
        
    def merge(self, source_df, condition):
        print(f"[Delta] MERGE INTO logic initiated.")
        print(f"[Delta] Condition: {condition}")
        return self
    
    def whenMatchedUpdateAll(self):
        print("[Delta] Registered: UPDATE existing records when condition matches.")
        return self
        
    def whenNotMatchedInsertAll(self):
        print("[Delta] Registered: INSERT new records when no match is found (CDC Upsert).")
        return self
        
    def execute(self):
        print("[Delta] Executing Transaction...")
        print("  > Writing new target Parquet files")
        print("  > Committing version 00000004.json to _delta_log")
        print("[Delta] Commit Successful. ACID guaranteed.\n")

print("--- Code Simulation Loading ---\n")

In [ ]:
# Production Pattern: The CDC MERGE (Upsert)
# In PySpark, this handles Change Data Capture natively without rewriting the entire Lake.

# 1. Load the target Delta Table
target_table = MockDeltaTable("s3://lakehouse-prod/customer_profiles/")

# 2. Perform the Upsert
target_table.merge(
    source_df="new_cdc_stream_data",
    condition="target.customer_id = source.customer_id"
).whenMatchedUpdateAll(  
).whenNotMatchedInsertAll( 
).execute()

## 3. Advanced Features: Time Travel and Vacuuming

Because Delta Lake retains older Parquet files and records them in historical JSON log commits, querying historical states is trivial.

In [ ]:
def query_time_travel(timestamp):
    print(f"[Spark] spark.read.format('delta').option('timestampAsOf', '{timestamp}').load('s3://...')")
    print(f"[Engine] Accessing _delta_log\n[Engine] Replaying commits up until {timestamp}...")
    print(f"[Engine] Ignoring Parquet files added AFTER {timestamp}. returning Snapshot DataFrame.\n")

query_time_travel("2026-02-26 14:00:00")

def run_vacuum(retention_hours=168):
    print(f"[Delta] Running VACUUM (Retention: {retention_hours} hours)")
    print("[Delta] Scanning _delta_log for files unreferenced in recent commits...")
    print("[Delta] Physically deleting orphaned part-000*.parquet files from S3 to save costs.")

run_vacuum(retention_hours=168) # 7 Days

## 4. Iceberg vs Delta Lake: The Showdown

| Feature | Delta Lake | Apache Iceberg |
|---------|------------|----------------|
| **Origin** | Databricks (2019) | Netflix (2018), Apache Foundation |
| **Primary Engine Focus** | Spark highly optimized, Databricks native | Spark, Flink, Trino, Presto, Athena natively |
| **Catalog Requirement** | Optional (Can use raw S3 path strictly) | Required (Glue, Hive Metastore, Nessie, REST) |
| **Schema Evolution** | Supported | Highly advanced (Columns tracked by explicit ID, not name) |
| **Hidden Partitioning** | No | Yes (Transforms dates implicitly, protecting consumer queries) |

## 5. The Medallion Architecture (Lakehouse)

```text
Kafka / Fivetran / APIs
        ↓
[ 🥉 Bronze ]  S3 Parquet / Delta  - Raw, Append-Only, unmodified historical ledger.
        ↓      (Spark / dbt transformations)
[ 🥈 Silver ]  S3 Delta / Iceberg  - Cleansed, deduplicated, Upserted (MERGE applied here). ACID.
        ↓      (Aggregations, Joins, Business Logic)
[ 🥇 Gold ]    S3 Delta / Iceberg  - Highly optimized, read-heavy, aggregated for dashboards.
        ↓
Athena / Redshift Spectrum / DuckDB (Query Engines)
```

## 6. Interview Q&A

**Q: What is the Delta transaction log and why is it important?**  
A: A JSON log directory representing every explicit transaction (insert, update, delete, schema change). It entirely enables ACID: readers natively see a consistent snapshot instantly, writers completely avoid corrupting in-flight reads natively. It fully unlocks time travel by replaying logs dynamically natively.

**Q: How would you handle a GDPR "right to be forgotten" precisely request in a data lake?**  
A: With plain raw Parquet: impossible practically without entirely completely natively physically rewriting thousands of files globally. With Delta/Iceberg: `DELETE FROM table WHERE user_id = 'X'` issues a row-level delete securely tracked natively inside the transaction log cleanly. You then formally execute `VACUUM` subsequently to unequivocally physically permanently obliterate the underlying targeted Parquet data records natively.

**Q: What is hidden partitioning in Iceberg exactly?**  
A: Iceberg entirely manages explicit partition transforms seamlessly strictly internally — end-user queries literally don't need to specifically know the distinct partition scheme explicitly. You confidently query by standard `date`, and Iceberg seamlessly handles mapping `date-bucket` resolution natively completely transparently. This brilliantly entirely structurally avoids the catastrophic "partition evolution" problem completely where altering native partition granularity breaks downstream queries entirely instantly.

## 7. The Citi Angle

**Modernizing Global Telemetry Storage**  
At Citi, storing massive server diagnostics exclusively strictly on vanilla HDFS Parquet created nightmares natively handling delayed explicitly late-arriving distinct logs. Every late metric strictly necessitated literally deleting entirely and explicitly totally recreating complete daily partitions explicitly identically manually. By migrating explicitly natively to Iceberg effectively, late-arriving metrics securely appended cleanly executing via natively guaranteed ACID isolated Upserts (`MERGE`), radically reducing exactly compute overhead strictly saving millions cleanly natively!